# STARCOP training on Google Colab (T4, Environment A)

Runs `src/training/train.py` -- the same entrypoint `train_mac.sh`/`train_desktop.sh` use -- on a free-tier Colab GPU. See `mlops-methane-detection-plan.md` TASK-3.3c for the full design and `internal-docs/setup/environment-notes.md`'s Colab sections for how every fix below was found.

**Before running anything else: Runtime → Change runtime type → Hardware accelerator → GPU (T4).** This isn't just about getting a GPU -- Colab's CPU-only runtime defaults to a newer Python that has no `torch==1.13.1` wheel at all. Picking the GPU runtime also switches to a Python version that does. Leaving this on CPU (or picking the wrong runtime) breaks the install cells below in a confusing way, not a clean error.

**Run cells top-to-bottom, in order, in a single kernel session.** Later cells depend on state earlier cells set (the cloned repo becoming the working directory, installed packages, `sys.path`) -- running a cell standalone, or after a kernel restart without re-running everything above it, produces confusing errors like `ModuleNotFoundError: No module named 'colab_bootstrap'` that look like a real bug but just mean an earlier cell's effect on this kernel is gone. If that happens, re-run from section 1.

## 1. Clone the repo

In [12]:
import os
from pathlib import Path

# Always reset to a known-good cwd first. A prior cell/session can leave this
# kernel's own process cwd pointing at a directory that no longer exists (e.g.
# after `rm -rf` on a directory the kernel happened to be sitting inside) --
# every subsequent `!` shell command inherits that broken cwd and fails at
# `shell-init: error retrieving current directory` before it even runs the
# given command, regardless of what absolute paths this cell itself uses.
os.chdir("/content")

REPO_DIR = Path("/content/methane-detection")
if not (REPO_DIR / ".git").exists():
    !git clone --recurse-submodules https://github.com/douglas-martins/methane-detection.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already cloned, skipping.")
%cd $REPO_DIR

/content/methane-detection already cloned, skipping.
/content/methane-detection


## 2. Dataset name

Defaults to `starcop_mini` (342 MB). `starcop_raw` (59 GB / 75k files) works but is slow on Colab's ephemeral disk, and each fresh session re-pulling it repeats the Google Drive API call volume flagged as a rate-limit risk (D-01) -- expect it to take a while, and avoid re-running this notebook top-to-bottom repeatedly against `starcop_raw` in a short window.

In [13]:
DATASET_NAME = "starcop_mini"

## 3. Install Environment A's pinned stack

Exact order confirmed by TASK-3.3c's step 1 spike -- reordering or combining these differently reintroduces bugs already found once (see `internal-docs/setup/environment-notes.md`):

1. `pip<24.1` -- Colab's default pip refuses `pytorch-lightning==1.6.4`'s malformed wheel metadata.
2. `numpy<2` -- must land before `torch` touches a real tensor, or torch's numpy interop breaks (`_ARRAY_API not found`).
3. `torch==1.13.1` -- has a `cp311` wheel on the GPU runtime's Python 3.11.13 (no separate Python provisioning needed).
4. The **complete** `vendor/starcop/requirements.txt` pin set -- not just the subset the step 1 spike's toy Lightning-only compute check needed. `train.py`'s real import chain (`starcop.data.dataset`, `starcop.data.datamodule`, etc.) needs `rasterio`/`geopandas`/`scikit-image`/`omegaconf`/`hydra-core` too, which the toy check never imported and so never caught missing -- found live when a real `train.py` run hit `ModuleNotFoundError: No module named 'rasterio'`. Plus `requirements/env-a-mlflow.txt`'s `mlflow<3.7`/`boto3`/`protobuf<4` (mlflow's `torch.export` import bug and wandb's old-style `_pb2.py` files, same reasons documented there). Plus `georeader-spaceml` (pinned to the same `2.3.4` `uv.lock` uses) -- listed in `vendor/starcop/requirements_package.txt` and this project's own `pyproject.toml`, but *not* in `vendor/starcop/requirements.txt` itself, so copying only the latter's pins misses it; `feature_extration.extract_features()` (called from `starcop_datamodule.prepare_data()`) imports it directly and only fails at real `train.py` runtime, same as the `rasterio` gap above.

In [14]:
!python -m pip install -q "pip<24.1"
!pip --version

DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
pip 24.0 from /usr/local/lib/python3.11/dist-packages/pip (python 3.11)


In [15]:
!pip install -q "numpy<2"
!pip install -q torch==1.13.1

DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.8.1 requires torch>=2.0.0, but you have torch 

In [35]:
!pip install -q \
  "pytorch-lightning==1.6.4" \
  "torchmetrics==0.10.0" \
  "kornia==0.6.7" \
  "wandb==0.13.3" \
  segmentation_models_pytorch \
  fsspec \
  gcsfs \
  omegaconf \
  hydra-core \
  ipython \
  ipykernel \
  rasterio \
  geopandas \
  matplotlib \
  scikit-image \
  scikit-learn \
  "setuptools<81" \
  "protobuf<4" \
  "mlflow<3.7" \
  boto3 \
  "georeader-spaceml==2.3.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.1/339.1 kB 8.8 MB/s eta 0:00:00:00:01
DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
asyncssh 2.24.0 requires cryptography>=48.0.1, but you have cryptography 46.0.7 which is incompatible.


In [17]:
# Colab preinstalls `transformers`, and torchmetrics==0.10.0's functional API
# unconditionally imports its BERTScore metric whenever `transformers` is merely
# importable (torchmetrics/functional/text/__init__.py's `if _TRANSFORMERS_AVAILABLE`
# only checks package *presence*, not that importing it actually works). That chain
# pulls in transformers.image_transforms, which unconditionally does `import
# tensorflow` -- and Colab's preinstalled tensorflow needs protobuf>=3.20.3, while
# the protobuf<4 pin above (needed for wandb==0.13.3's old-style _pb2.py files)
# resolves to protobuf==3.19.6 in this combined install. Result: `import
# pytorch_lightning` (which imports torchmetrics) crashes train.py with
# `ImportError: cannot import name 'builder' from 'google.protobuf.internal'`,
# even though STARCOP training never touches BERTScore or tensorflow. Uninstalling
# transformers removes it from torchmetrics' availability check entirely, so that
# whole broken import chain is never attempted.
!pip uninstall -y transformers

Found existing installation: transformers 4.53.2
Uninstalling transformers-4.53.2:
  Successfully uninstalled transformers-4.53.2


## 4. Install DVC

Same version this project pins (`uv.lock`), plus the two fixes TASK-3.3c's step 1b spike found: `protobuf<4` must be pinned in the *same* command as the `dvc[gdrive]` install (a separate later command re-resolves it upward and breaks wandb/pytorch-lightning again), and `pyOpenSSL` has to be removed entirely rather than upgraded or downgraded -- `pydrive2`'s legacy `oauth2client` auth path has no single `pyOpenSSL`/`cryptography` version pairing that satisfies both `cryptography`'s newer API and `oauth2client`'s `crypto.sign()` call. Removing `pyOpenSSL` forces `oauth2client` onto its pure-Python RSA signer instead, which has no C-extension ABI to break.

In [18]:
!pip install -q "dvc[gdrive]==3.67.1" "protobuf<4"
!pip uninstall -y pyOpenSSL
!pip install -q rsa pyasn1-modules

DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.6.0 requires cryptography<47,>=43.0.0, but you have cryptography 50.0.0 which is incompatible.
Found existing installation: pyOpenSSL 22.0.0
Uninstalling pyOpenSSL-22.0.0:
  Successfully uninstalled pyOpenSSL-22.0.0
DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or c

## 5. Credentials

**In the Colab web UI** (colab.research.google.com): one-time setup, per Google account -- click the 🔑 key icon in the left sidebar → **Secrets**, and add each of the following, with **Notebook access** enabled. None of these are ever written into this `.ipynb` file.

**Running via VS Code's Colab connection instead**: `google.colab.userdata.get()` can't reach the Secrets panel from there (no browser frontend for its RPC) -- and a `Colab: Open Terminal` shell is a *separate process* from this kernel, so `export`ing a variable there never reaches `os.environ` here either. Use a file instead, loaded directly into this kernel's own process:

- Create `.env.mlflow` in the cloned repo root (`/content/methane-detection/.env.mlflow`, git-ignored -- the same file/convention `train_mac.sh`/`train_desktop.sh` already use) with the five `MLFLOW_*`/`AWS_*` lines below marked required, `KEY=value` per line, plus `WANDB_API_KEY=...` if wanted. Easiest via VS Code's file explorer, since it's browsing this same Colab VM's filesystem.
- For `DVC_GDRIVE_SERVICE_ACCOUNT_JSON`, don't put the JSON blob in `.env.mlflow` -- place the key file itself directly at `/content/gdrive-service-account.json` (drag it in via VS Code's file explorer, or copy the same key file already used by the `mac-mps` worker). The next cell checks for that file before asking for a secret/env var at all.

| Secret / env var name | Value |
|---|---|
| `MLFLOW_TRACKING_URI` (informational only) | Always hardcoded to `https://methane-detection-mlflow.ghostface.tech` by the next cell regardless of what's set here -- no need to add it as a Secret/env var, it's just documented for clarity |
| `MLFLOW_TRACKING_USERNAME` | MLflow basic-auth username |
| `MLFLOW_TRACKING_PASSWORD` | MLflow basic-auth password |
| `MLFLOW_S3_ENDPOINT_URL` | B2 artifact store endpoint |
| `AWS_ACCESS_KEY_ID` | Dedicated client-side B2 Application Key ID (see `internal-docs/setup/environment-notes.md`) |
| `AWS_SECRET_ACCESS_KEY` | That key's secret |
| `DVC_GDRIVE_SERVICE_ACCOUNT_JSON` | Colab web UI only -- full contents of the existing D-01 service-account JSON key (**reuse the key already created for the `mac-mps` worker**, don't mint a new one). Via VS Code, place the key file directly instead (see above). |
| `WANDB_API_KEY` (optional) | Only if you want real W&B logging from Colab; otherwise training runs with `WANDB_MODE=disabled` (D-09) and MLflow logging is unaffected |

In [19]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "training" / "colab_bootstrap.py").exists():
    raise RuntimeError(
        f"src/training/colab_bootstrap.py not found under {REPO_ROOT}. Run section 1's "
        "clone cell first, or check this kernel's working directory with `!pwd` -- it "
        "must be the cloned methane-detection repo root. This usually means an earlier "
        "cell was never run in this kernel session (or the kernel restarted)."
    )
sys.path.insert(0, str(REPO_ROOT / "src" / "training"))

# Always load .env.mlflow if present, regardless of whether google.colab itself is
# importable -- read_secret()'s environ fallback needs something to fall back TO.
# This matters even when google.colab *is* importable: a VS Code-attached kernel is
# a genuine Colab VM, so the import can succeed there too, but userdata.get() still
# fails at call time (no browser frontend to service its RPC). read_secret() already
# catches that and falls back to environ -- but only if .env.mlflow was loaded here.
!pip install -q python-dotenv
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env.mlflow")

try:
    from google.colab import userdata

    userdata_get = userdata.get
except ImportError:
    # No google.colab module at all -- e.g. running outside Colab entirely.
    userdata_get = None

import colab_bootstrap  # noqa: E402
import launch_profiles  # noqa: E402

DVC_JSON_SECRET = "DVC_GDRIVE_SERVICE_ACCOUNT_JSON"

# Fail loudly with the missing credential's name, same as train_mac.sh/train_desktop.sh's
# pre-flight check -- don't let a missing credential surface later as an opaque subprocess
# failure. DVC_JSON_SECRET is skipped here: cell below checks for an already-placed key
# file before falling back to a secret/env var for it.
for secret_name in colab_bootstrap.required_colab_secrets():
    if secret_name == DVC_JSON_SECRET:
        continue
    value = colab_bootstrap.read_secret(secret_name, userdata_get, os.environ)
    if not value:
        raise RuntimeError(
            f"Required credential {secret_name!r} is not set. In the Colab web UI, add "
            "it via the \U0001f511 key icon in the left sidebar (Notebook access enabled). "
            "Via VS Code, add it to .env.mlflow in the repo root instead (see the markdown "
            "cell above)."
        )
    os.environ[secret_name] = value

# Public, non-secret -- hardcoded rather than trusted from a Secret/env var, same
# reasoning as train_mac.sh/train_desktop.sh's MLFLOW_TRACKING_URI.
os.environ["MLFLOW_TRACKING_URI"] = "https://methane-detection-mlflow.ghostface.tech"

print("Credentials loaded (values not printed).")

DEPRECATION: pytorch-lightning 1.6.4 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Credentials loaded (values not printed).


In [20]:
key_path = Path("/content/gdrive-service-account.json")
if key_path.exists():
    print(f"Using existing service-account key already at {key_path}.")
else:
    key_json = colab_bootstrap.read_secret(DVC_JSON_SECRET, userdata_get, os.environ)
    if not key_json:
        raise RuntimeError(
            f"{DVC_JSON_SECRET} is not set, and no key file already exists at {key_path}. "
            "In the Colab web UI, add it as a Secret. Via VS Code, place the key file "
            f"directly at {key_path} instead (see the markdown cell above) -- simpler "
            "than putting the JSON blob in .env.mlflow as a single line."
        )
    key_path.write_text(key_json)
os.chmod(key_path, 0o600)

for command in colab_bootstrap.dvc_service_account_setup_commands(str(key_path)):
    subprocess.run(command, check=True)

print("DVC configured for the service account (no OAuth prompt expected).")

Using existing service-account key already at /content/gdrive-service-account.json.
DVC configured for the service account (no OAuth prompt expected).


In [21]:
# D-09 parity with train_mac.sh/train_desktop.sh: default to WANDB_MODE=disabled
# when no WANDB_API_KEY secret was set, so training doesn't hang on an interactive
# wandb.login() prompt with no TTY. An explicit WANDB_MODE or a real key both win.
wandb_mode = colab_bootstrap.resolve_wandb_mode(
    wandb_api_key=os.environ.get("WANDB_API_KEY"),
    requested_mode=os.environ.get("WANDB_MODE"),
)
if wandb_mode is not None:
    os.environ["WANDB_MODE"] = wandb_mode
print("WANDB_MODE:", os.environ.get("WANDB_MODE", "<unset, real key present>"))

WANDB_MODE: disabled


## 6. Pull the dataset

`data/processed/<dataset>` is **not** a valid pull target -- it's only the parent of `dvc.yaml`'s five separate stage outputs, confirmed by TASK-3.3c's step 1b spike (`NoOutputOrStageError`). Pull the raw dataset directly, then only the three processed-pipeline stages `train.py` actually reads: `normalize` (-> `selected`, the real raster files the patch/split CSVs' `folder` column points into), `split` (-> `splits/*.csv`), and `patch_extract` (-> `patches/*.csv`). `stats@`/`coordinates@` are dvc.yaml stages too, but nothing under `src/training/` reads them -- only their own preprocessing scripts/tests do -- so they're intentionally left out here.

In [32]:
!dvc pull data/{DATASET_NAME} -v

2026-08-25 00:02:16,735 DEBUG: v3.67.1 (pip), CPython 3.11.13 on Linux-6.6.122+-x86_64-with-glibc2.35
2026-08-25 00:02:16,735 DEBUG: command: /usr/local/bin/dvc pull data/starcop_mini -v
2026-08-25 00:02:17,217 DEBUG: Assuming 'data/starcop_mini' to be a stage inside 'dvc.yaml'
2026-08-25 00:02:17,491 DEBUG: Checking if stage 'data/starcop_mini' is in 'dvc.yaml'
2026-08-25 00:02:21,031 DEBUG: Preparing to transfer data from 'gdrive://1i9YiEbj-d3gPd9oy5rgoJsKbQDSxmWhq/files/md5' to '/content/methane-detection/.dvc/cache/files/md5'
2026-08-25 00:02:21,032 DEBUG: Preparing to collect status from '/content/methane-detection/.dvc/cache/files/md5'
2026-08-25 00:02:21,032 DEBUG: Collecting status from '/content/methane-detection/.dvc/cache/files/md5'
Fetching
!
  0% |          |0/? [00:00<?,    ?files/s]
Fetching                                   
2026-08-25 00:02:21,042 DEBUG: Assuming 'data/starcop_mini' to be a stage inside 'dvc.yaml'
Building workspace index          |438 [00:00, 15.4kent

In [33]:
!dvc pull \
  "normalize@{DATASET_NAME}" \
  "split@{DATASET_NAME}" \
  "patch_extract@{DATASET_NAME}" \
  -v

2026-08-25 00:03:55,028 DEBUG: v3.67.1 (pip), CPython 3.11.13 on Linux-6.6.122+-x86_64-with-glibc2.35
2026-08-25 00:03:55,028 DEBUG: command: /usr/local/bin/dvc pull normalize@starcop_mini split@starcop_mini patch_extract@starcop_mini -v
2026-08-25 00:03:55,463 DEBUG: Assuming 'normalize@starcop_mini' to be a stage inside 'dvc.yaml'
2026-08-25 00:03:55,586 DEBUG: Assuming 'split@starcop_mini' to be a stage inside 'dvc.yaml'
2026-08-25 00:03:55,650 DEBUG: Assuming 'patch_extract@starcop_mini' to be a stage inside 'dvc.yaml'
2026-08-25 00:03:59,336 DEBUG: Preparing to transfer data from 'gdrive://1i9YiEbj-d3gPd9oy5rgoJsKbQDSxmWhq/files/md5' to '/content/methane-detection/.dvc/cache/files/md5'
2026-08-25 00:03:59,337 DEBUG: Preparing to collect status from '/content/methane-detection/.dvc/cache/files/md5'
2026-08-25 00:03:59,337 DEBUG: Collecting status from '/content/methane-detection/.dvc/cache/files/md5'
Fetching
!
  0% |          |0/? [00:00<?,    ?files/s]
Fetching                   

## 7. Train

Same `launch_profiles.build_launch_args` function `train_mac.sh`/`train_desktop.sh` use, so the `colab` machine profile (`training.accelerator=gpu training.devices=1`) stays a single source of truth across all three launch paths.

In [37]:
launch_args = launch_profiles.build_launch_args("colab", DATASET_NAME)
print("train.py", *launch_args)

result = subprocess.run(
    [sys.executable, "src/training/train.py", *launch_args],
    capture_output=True,
    text=True,
)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
print("exit code:", result.returncode)

train.py +machine=colab +dataset_name=starcop_mini training.accelerator=gpu training.devices=1
03,029][botocore.credentials][INFO] - Found credentials in environment variables.
[2026-08-25 00:36:03,070][botocore.credentials][INFO] - Found credentials in environment variables.
[2026-08-25 00:36:03,072][botocore.credentials][INFO] - Found credentials in environment variables.
[2026-08-25 00:36:03,102][botocore.credentials][INFO] - Found credentials in environment variables.
[2026-08-25 00:36:35,128][__main__][WARNING] - run_validation (train data) failed -- skipping, see TASK-2.2 note
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper